In [1]:
from datasets import load_dataset
from tqdm import tqdm
import sys
from pathlib import Path
import json
import os

PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
MODELS_PATH = PROJECT_ROOT / "models"
DATA_PROCESSED_PATH = SRC_PATH / "data" / "processed"
TRAIN_DATASET = DATA_PROCESSED_PATH / "train_cats_dogs"
VALIDATION_DATASET = DATA_PROCESSED_PATH / "validation_cats_dogs"

if not os.path.exists(str(DATA_PROCESSED_PATH)):
    os.makedirs(str(DATA_PROCESSED_PATH))

sys.path.append(str(SRC_PATH))

from data.dataset import CatsDogsDataset
from data.transforms import get_train_transforms

## **Limpieza del dataset**

In [2]:
dataset = load_dataset("microsoft/cats_vs_dogs", split="train")

def is_valid(example):
    try:
        example["image"].convert("RGB")
        return True
    except Exception:
        return False

clean_dataset = dataset.filter(is_valid)

print(f"Dataset original: {len(dataset)} imágenes")
print(f"Dataset limpio: {len(clean_dataset)} imágenes")

Dataset original: 23410 imágenes
Dataset limpio: 23410 imágenes


## **División del dataset**

In [3]:
splits = clean_dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="labels")

train = splits["train"]
validation = splits["test"]

## **Almacenamiento del dataset de entrenamiento y validación**

In [4]:
train.save_to_disk(str(TRAIN_DATASET))
validation.save_to_disk(str(VALIDATION_DATASET))

Saving the dataset (0/2 shards):   0%|          | 0/18728 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4682 [00:00<?, ? examples/s]

## **Dataset para pytorch desacoplado de la fuente original**

In [5]:
train_pytorch_dataset = CatsDogsDataset(train, transform=get_train_transforms())

img, label = train_pytorch_dataset[0]
img.shape, label

(torch.Size([3, 224, 224]), 1)

## **Archivo json con las categorías del dataset mapeadas**

In [6]:
label_names = clean_dataset.features["labels"].names
JSON_PATH = MODELS_PATH / "label_mapping.json"

with open(str(JSON_PATH), "w") as f:
    json.dump({"labels":label_names}, f)